In [ ]:
import torch
from torch import nn
from parameter import Parameter

In [3]:
p = Parameter(torch.randn(3))

assert p.requires_grad
assert p.is_leaf                    # or .grad never gets populated
assert type(p * 2) is torch.Tensor  # subclass must not leak into activations
print("ok")

ok


In [4]:
d = torch.randn(2, 3)

def probe(p):
    (p * 2).sum().backward()
    g, p.grad = p.grad.clone(), None
    return {
        "type": type(p).__name__,
        "requires_grad": p.requires_grad,
        "is_leaf": p.is_leaf,  # else .grad never fills in
        "values == d": torch.equal(p.detach(), d),
        "shares storage": p.data_ptr() == d.data_ptr(),
        "type after op": type(p * 2).__name__,  # must be Tensor, not Parameter
        "grad": tuple(g.flatten()[:2].tolist()),
        "grad type": type(g).__name__,
    }

mine, ref = probe(Parameter(d)), probe(nn.Parameter(d))
for k in mine:
    print(
        f"{k:15} : {mine[k]!s:>20} | {ref[k]!s:>20}",
        "" if mine[k] == ref[k] else "<-- DIFFERS",
    )

type            :            Parameter |            Parameter 
requires_grad   :                 True |                 True 
is_leaf         :                 True |                 True 
values == d     :                 True |                 True 
shares storage  :                 True |                 True 
type after op   :               Tensor |               Tensor 
grad            :           (2.0, 2.0) |           (2.0, 2.0) 
grad type       :               Tensor |               Tensor 
